# Molecular Dynamics

This Project is based on Chapter 8 from the book: An Introduction to Computer Simulation Methods Third Edition (revised)

https://www.compadre.org/osp/items/detail.cfm?ID=7375

Molecular dynamics is the study of how atoms or even molecules behave. For this project, I choosen to study molecular dynamics with periodic boundaries. Instead of having hard boundary conditions like say the walls of a piston containing some gas. We wanted to measure the behavior of a gas by its self with no boundaries imposed on it.

In other words, if we were to model the dynamics of say a liquid or a solid that not only comprimises a couple of atoms but trillions and so forth, we do not have the capabilities to model such a system, nor do we have enough space to store all of it's position and velocities. So a simple solution is to consider a central cell with a fix length and height, in our case we choose to stay in 2D to make the calculations simplier, and cosinder what happens if we allowed the neighboring cells to look exactly the same as the central cell. So when a particle from our central cell moved passed a boundary, a new particle would emerge from the other side of the boundary. This would be the same as the geometry of a torus. We would then calculate the force between the each atom or molecule in the central cell and update their positions and velocities.

Now the key points of the implementation are how do we go about doing this? First and for most, we note that chooce to use the Lenard-Jones potential to model the behavior of atoms/molecules as they are very close to each other, repulsive force, and far from each other, slightly attracted to each other.

The numerical algorithm that we implement is the Verlet Algorithm:

$$
x_{n+1} = x_{n} + v_{n} \Delta t + \frac{1}{2} a_{n} (\Delta t)^2
$$
$$
v_{n+1} = v_n +\frac{1}{2}(a_{n+1} + a_{n} )\Delta t
$$

In the code the implementation is rather straight forward, first calculate the initial forces and thereby the accelerations each particle/atom experiences at a previous time step $n$. We then use those accelerations find the positions at $n+1$. For which we can then find the new accelerations at $n+1$. Finally with the acceleration from $n$ and $n+1$ we find the velocity of the particles at $n+1$ as shown above.

On the topic of Preriodic boundary conditions and forces. If we image a truly periodic boundary, then molecules could have a shorter distance between them if we consider thier distance accros the boundary. For example, if we have to two particles that are a distance $\frac{3}{4} L$ apart, then accross the boundary these particles would be a distance $\frac{1}{4} L$ apart. So the problem is which distance should we consider to calculate the force between the particles. The answer is the shortest distance! To make sense of this, what we are implying is a cut of radius, meaning that the distance that two particles can interact is $L/2$. What this allows us is to avoid double counting, in other words, we don't wish to consider the force between the real particles in the central cell and their copy images on the neighboring cells. Therefore by apply the cut out radius we avoid having this issues. Moreover, the Length of the box should be sufficient such that the forces at half of it's length are so impercetable small that they can be ignored.

On the topic of initial conditions such as the starting velocities of the particles. We may wish to choose completely random velocities for all particles and then measure the temperature of the system after equilibrium sets in. However, we can scale the velocities of the particles choose at random such that it's total Kinetic energy equals the desired temperature of the system. To that end we use the equipartition theorem, the mean kinetic energy of a particle per degree of freedom is $kT/2$, where $k$ is the boltzmann's constant and $T$ is the temperature. We generalize this relation to define the temperature at time $t$ by:
$$
k T(t) = \frac{2}{d} \frac{K(t)}{N} = \frac{1}{Nd} \sum_{i=1}^{N} m_i v_i(t)\cdot v_i(t)
$$
where $K$ is the total Kinetic Energy of the system, $v_i$ is the velocity of the particle $i$ with mass $m_i$, and $d$ is the spatial dimension of the system. We choose the particles velcoties at random and then resacle them to reach the correct Kinetic energy of the system.

Lastly, on the position of the particles, we may just choose to have particles be positioned at random, however, if two particles are places very close to each other, their repulsive force could be incredibly large, wich would lead to breaking the our numerical integration. So we must be careful how we place the particles on the cell. One option is to just choose at random and check the distance between two particles against each other, making sure that their are a certain distance apart and if they fail to reach that distance, then we place them at random again until we place all atoms. As you can see this step can be very computationally expensive. Another more viable option is to place each atoms in a lattice structure witha predfined distance between them. If the internal kinetic energy of the system, in other words the velocity of the system is very large, the this would be similar to a solid melting away. This is in fact the way we have definite the starting position of our particles on the cell.

Now please take a look at the code below that caculate the molecular interactions with periodic boundary conditions:



In [ ]:
import numpy as np
import math

# Molecular Dynamics System Class!
# To organize the system in a way that is both clear to understand
# but yet capable to encapsulate all our variables and functions
# we choose to make a class!

class MDSystem:

    # Initialization of the class:
    def __init__(self, num_particles=100 , box_length=10, initial_temperature = 50):

        # Starting Conditions!
        self.N = num_particles
        self.L = box_length  # For the time being we assume that the box is a square
        self.target_temp = initial_temperature  # The target temperature will be used for the velocity of the system.
        self.t = 0 # starting time

        # State arrays, position, velocity and acceleration!
        self.x = np.zeros(self.N)
        self.y = np.zeros(self.N)
        self.vx = np.zeros(self.N)
        self.vy = np.zeros(self.N)
        self.ax = np.zeros(self.N)
        self.ay = np.zeros(self.N)

        # Thermodynamic Variables
        self.steps = 0
        self.potential_energy = 0.0
        self.virial = 0.0
        self.kinetic_energy = 0.0

        # Accumulators
        self.ke_accumulator = 0.0
        self.ke_sq_accumulator = 0.0
        self.pe_accumulator = 0.0
        self.virial_accumulator = 0.0

        # Initialize the central cell, position and velocities!
        self._initialize_lattice()
        self._initialize_velocities()
        self.compute_forces() # Calculate initial forces before the first step!

    def reset(self):
        """Resets the thermodynamic accumulators and step count."""
        # Reset All variables
        self.x = np.zeros(self.N)
        self.y = np.zeros(self.N)
        self.vx = np.zeros(self.N)
        self.vy = np.zeros(self.N)
        self.ax = np.zeros(self.N)
        self.ay = np.zeros(self.N)
        self.t = 0
        self.potential_energy = 0.0
        self.virial = 0.0
        self.kinetic_energy = 0.0
        self.steps = 0
        self.ke_accumulator = 0.0
        self.ke_sq_accumulator = 0.0
        self.pe_accumulator = 0.0
        self.virial_accumulator = 0.0

        # Initialize the central cell, position and velocities!
        self._initialize_lattice()
        self._initialize_velocities()
        self.compute_forces() # Calculate initial forces before the first step!


    def _initialize_lattice(self):
        """Places particles on a simple square lattice to avoid overlapping."""
        n_side = math.ceil(math.sqrt(self.N))
        spacing = self.L / n_side

        # Updating position variables!
        particle_idx = 0
        for i in range(n_side):
            for j in range(n_side):
                if particle_idx < self.N:
                    self.x[particle_idx] = (i + 0.5) * spacing
                    self.y[particle_idx] = (j + 0.5) * spacing
                    particle_idx += 1

    def _initialize_velocities(self):
        """Assigns random velocities and scales them to the target temperature."""
        self.vx = np.random.uniform(-0.5, 0.5, self.N)
        self.vy = np.random.uniform(-0.5, 0.5, self.N)

        # Remove center-of-mass momentum (prevent the whole box from drifting)
        self.vx -= np.mean(self.vx)
        self.vy -= np.mean(self.vy)

        # Scale to target temperature: T = <v^2> / 2 (for 2D)
        current_temp = np.mean(self.vx**2 + self.vy**2) / 2.0 # Same as 0.5*v2sum/N
        scale_factor = math.sqrt(self.target_temp / current_temp)

        self.vx *= scale_factor
        self.vy *= scale_factor

    def compute_forces(self):
        """Calculate Lennard-Jones forces and the virial."""
        self.ax.fill(0.0)
        self.ay.fill(0.0)
        self.potential_energy = 0.0
        self.virial = 0.0

        rc2 = (self.L / 2.0)**2 # Cutoff radius squared

        # The i < j loop prevents double counting!
        for i in range(self.N):
            for j in range(i + 1, self.N):
                dx = self.x[i] - self.x[j]
                dy = self.y[i] - self.y[j]

                # pbcSeparation: Minimum Image Convention
                dx = dx - self.L * round(dx / self.L)
                dy = dy - self.L * round(dy / self.L)

                r2 = dx*dx + dy*dy

                if r2 < rc2:
                    oneOverR2 = 1.0 / r2
                    oneOverR6 = oneOverR2 * oneOverR2 * oneOverR2

                    # Save the Lennard-Jones Potential: V = 4(r^-12 - r^-6)!
                    self.potential_energy += 4.0 * oneOverR6 * (oneOverR6 - 1.0)

                    # Calculate the Force magnitude / r: F/r = 48/r^2 * (r^-12 - 0.5*r^-6)
                    f_over_r = 48.0 * oneOverR2 * oneOverR6 * (oneOverR6 - 0.5)

                    # Individual Force Components
                    fx = f_over_r * dx
                    fy = f_over_r * dy

                    # Newton's 3rd Law!
                    self.ax[i] += fx
                    self.ay[i] += fy
                    self.ax[j] -= fx
                    self.ay[j] -= fy

                    # Accumulate virial for pressure! This will come in handy later!
                    self.virial += (dx * fx) + (dy * fy)

    def velocity_verlet(self, dt):
        """Appendix 3A: Advances the simulation by one time step dt."""
        # Step 1: Update positions and half-step velocities
        self.x += self.vx * dt + 0.5 * self.ax * dt**2
        self.y += self.vy * dt + 0.5 * self.ay * dt**2
        self.vx += 0.5 * self.ax * dt
        self.vy += 0.5 * self.ay * dt

        # Keep particles inside the primary simulation box [0, L]
        self.x = self.x % self.L
        self.y = self.y % self.L

        # Step 2: Recalculate forces based on NEW positions
        self.compute_forces()

        # Step 3: Complete the velocity update using the NEW forces
        self.vx += 0.5 * self.ax * dt
        self.vy += 0.5 * self.ay * dt

    def step(self, dt):
        """The main execution method for the physics engine."""
        self.velocity_verlet(dt)

        # Measure thermodynamic quantities
        self.kinetic_energy = 0.5 * np.sum(self.vx**2 + self.vy**2)

        # Add to accumulators
        self.ke_accumulator += self.kinetic_energy
        self.ke_sq_accumulator += self.kinetic_energy**2
        self.pe_accumulator += self.potential_energy
        self.virial_accumulator += self.virial

        self.steps += 1
        self.t += dt

    def get_state(self):
        """Returns the current state of the system."""
        return self.x, self.y, self.vx, self.vy


# ==========================================
# Run the Simulation
# ==========================================
if __name__ == "__main__":
    system = MDSystem(num_particles=64, box_length=12.0, initial_temperature=1.0)
    dt = 0.005

    print("Starting simulation...")
    for _ in range(1000):
        system.step(dt)

    # Calculate Averages (Section 8.7)
    avg_ke = system.ke_accumulator / system.steps
    avg_pe = system.pe_accumulator / system.steps
    avg_virial = system.virial_accumulator / system.steps

    # In 2D, T = <KE> / N
    temperature = avg_ke / system.N

    # Equation 8.9: Pressure = rho*T + W/(2*V)
    density = system.N / (system.L**2)
    pressure = density * temperature + avg_virial / (2.0 * system.L**2)

    print(f"Total Steps: {system.steps}")
    print(f"Average Temperature: {temperature:.3f}")
    print(f"Average Pressure:    {pressure:.3f}")
    print(f"Average Total Energy:{avg_ke + avg_pe:.3f}")


: 

In [ ]:
import matplotlib.pyplot as plt

# Get the current state of the system directly from attributes
x, y, vx, vy = system.x, system.y, system.vx, system.vy

# Create the scatter plot
plt.figure(figsize=(8, 8))
plt.scatter(x, y, s=100, c='royalblue', alpha=0.8, edgecolors='k')

# Set the axes limits to represent the simulation box
plt.xlim(0, system.L)
plt.ylim(0, system.L)

# Add labels and title
plt.title(f"MD Simulation Particle Positions (Step {system.steps})", fontsize=16)
plt.xlabel("X Position", fontsize=14)
plt.ylabel("Y Position", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)

# Display the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

# Initialize a fresh system for the animation
anim_system = MDSystem(num_particles=64, box_length=12.0, initial_temperature=1.0)
dt = 0.005

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, anim_system.L)
ax.set_ylim(0, anim_system.L)
ax.set_title("Molecular Dynamics Animation")
ax.set_xlabel("X Position")
ax.set_ylabel("Y Position")
ax.grid(True, linestyle='--', alpha=0.6)

# Create the initial scatter plot
scatter = ax.scatter(anim_system.x, anim_system.y, s=100, c='royalblue', edgecolors='k', alpha=0.8)

def update(frame):
    # Advance the simulation a few steps per frame for a smoother/faster visual
    for _ in range(5):
        anim_system.step(dt)

    # Update the positions in the scatter plot
    scatter.set_offsets(np.c_[anim_system.x, anim_system.y])
    ax.set_title(f"Molecular Dynamics (Step {anim_system.steps})")
    return scatter,

# Create the animation (100 frames, 50ms interval between frames)
ani = animation.FuncAnimation(fig, update, frames=100, interval=50, blit=False)

# Close the static figure so it doesn't display twice
plt.close(fig)

# Display the animation as an interactive HTML5 video element
HTML(ani.to_jshtml())

Conclusion:
This project helped understand the basics of molecular dynamics simulations! I hope that I can continue to learn more about this topic and eventually be able to apply this to real systems and model real materials. I hope that this brief introduction will also motivate other to explor this topic further. I hope to explor this topic further and be able to make a MD library and a nice GUI for it!